# WorldView Acquisition Search for Gus Pearson and SERC

Searches CSDA CMR for cloud-free WV2 8-band multispectral and panchromatic L1B acquisitions, screens metadata for off-nadir angle below 15 degrees, and ranks candidates by proximity to the G-LiHT lidar dates. Gus Pearson targets May-June near March 2013; SERC targets leaf-on June-August near July 2017 and also searches for a metadata-confirmed snow-free November-March pair. A cloud-free, near-nadir WV3 SWIR acquisition is included as an optional Gus Pearson addition.

The Gus Pearson bounding box is an approximate Flagstaff-area GPNA extent carried over from the older notebook; replace it with an authoritative site AOI if available. The SERC extent is the existing site bounding box. CMR does not index off-nadir or snow-free criteria as query parameters, so those are checked against each granule's UMM metadata. Winter candidates without an explicit snow-free metadata value are excluded.

In [ ]:
from __future__ import annotations

import calendar
import csv
import getpass
import html.parser
import json
import posixpath
import re
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import boto3
import requests
from botocore.config import Config

CMR_GRANULES_URL = "https://cmr.earthdata.nasa.gov/search/granules.json"
CMR_CONCEPT_URL = "https://cmr.earthdata.nasa.gov/search/concepts"
S3_CREDENTIALS_URL = "https://data.csdap.earthdata.nasa.gov/s3credentials"

COLLECTIONS = {
    "WV2_MSI": "C2497404794-CSDA",       # WorldView-2 8-band multispectral L1B
    "WV2_PAN": "C2497398128-CSDA",       # WorldView-2 panchromatic L1B
    "WV3_SWIR": "C2497415947-CSDA",      # WorldView-3 8-band SWIR L1B
}
FIRST_DATA_YEAR = 2009
LAST_DATA_YEAR = 2026
MAX_CLOUD_COVER = 0.0
MAX_OFF_NADIR_DEGREES = 15.0
PAGE_SIZE = 2000
OUTPUT_DIR = Path("/explore/nobackup/people/pmontesa/WV2_lidar_site_acquisitions")
MANIFEST_PATH = OUTPUT_DIR / "worldview_lidar_site_downloads.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# WGS84 bounding boxes [west, south, east, north]. Gus Pearson uses the older
# GPNA/Flagstaff regional extent from this notebook; replace with an exact AOI
# when one is available. The SERC extent is the existing site footprint extent.
SITES = {
    "Gus Pearson": {
        "bbox": (-111.920, 35.123, -111.570, 35.423),
        "lidar_date": date(2013, 3, 1),
        "primary_months": {5, 6},
        "optional_wv3_swir": True,
    },
    "SERC": {
        "bbox": (-76.736, 38.740, -76.386, 39.040),
        "lidar_date": date(2017, 7, 15),
        "primary_months": {6, 7, 8},
        "optional_wv3_swir": False,
    },
}
WINTER_MONTHS = {11, 12, 1, 2, 3}
DOWNLOAD_SUFFIXES = {".ntf", ".tif", ".tiff", ".xml", ".rpb", ".imd"}

## 1. Search and rank acquisition candidates
Searches the WV2 MSI and PAN collections across the archive for each site, then applies season, zero-cloud, metadata-confirmed off-nadir below 15 degrees, and (for SERC winter) snow-free filters. It selects the same acquisition signature for paired MSI/PAN and ranks complete pairs by distance to the lidar date. A separate best eligible WV3 SWIR candidate is considered for Gus Pearson.

In [ ]:
def as_float(value):
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, list):
        for part in value:
            parsed = as_float(part)
            if parsed is not None:
                return parsed
    if value is not None:
        match = re.search(r"[-+]?\d+(?:\.\d+)?", str(value))
        if match:
            return float(match.group())
    return None


def named_metadata_values(node, inherited_name=""):
    """Yield normalized names and values, retaining names through UMM lists."""
    if isinstance(node, dict):
        name = node.get("Name") or node.get("name") or inherited_name
        value = node.get("Values", node.get("Value", node.get("value")))
        if name and value is not None:
            yield re.sub(r"[^A-Z0-9]", "", str(name).upper()), value
        for key, child in node.items():
            child_name = name if key in {"Values", "Value", "value"} else str(key)
            yield from named_metadata_values(child, child_name)
    elif isinstance(node, list):
        for child in node:
            yield from named_metadata_values(child, inherited_name)


def attribute_number(umm, token):
    for name, value in named_metadata_values(umm):
        if token in name:
            number = as_float(value)
            if number is not None:
                return number
    return None


def snow_free_status(umm):
    for name, value in named_metadata_values(umm):
        if "SNOW" not in name:
            continue
        text = " ".join(map(str, value if isinstance(value, list) else [value])).strip().upper()
        if text in {"0", "0.0", "NO", "NONE", "FALSE", "SNOW FREE", "SNOW_FREE"}:
            return "snow-free"
        if text:
            return "snow-present-or-uncertain"
    return "unknown-review-needed"


def load_granule_umm(granule_id):
    response = requests.get(f"{CMR_CONCEPT_URL}/{granule_id}.umm_json", timeout=60)
    response.raise_for_status()
    items = response.json().get("items", [])
    if not items or "umm" not in items[0]:
        raise ValueError(f"CMR returned no UMM metadata for granule {granule_id}")
    return items[0]["umm"]


def month_window(year, month):
    last_day = calendar.monthrange(year, month)[1]
    return f"{year}-{month:02d}-01T00:00:00Z,{year}-{month:02d}-{last_day:02d}T23:59:59Z"


def search_site_collection(bbox, collection_id, months):
    """Search only requested months across the archive and page through CMR results."""
    granules_by_id = {}
    session = requests.Session()
    for year in range(FIRST_DATA_YEAR, LAST_DATA_YEAR + 1):
        for month in sorted(months):
            page_num = 1
            params = {
                "collection_concept_id": collection_id,
                "page_size": PAGE_SIZE,
                "bounding_box": ",".join(map(str, bbox)),
                "temporal": month_window(year, month),
                "cloud_cover": f",{MAX_CLOUD_COVER:g}",
            }
            while True:
                params["page_num"] = page_num
                response = session.get(CMR_GRANULES_URL, params=params, timeout=60)
                response.raise_for_status()
                entries = response.json().get("feed", {}).get("entry", [])
                if not entries:
                    break
                for granule in entries:
                    granules_by_id[granule["id"]] = granule
                hits = int(response.headers.get("CMR-Hits", len(entries)))
                if page_num * PAGE_SIZE >= hits or len(entries) < PAGE_SIZE:
                    break
                page_num += 1
                time.sleep(0.2)
    return list(granules_by_id.values())


def acquisition_key(granule):
    producer_id = granule.get("producer_granule_id") or granule.get("title") or ""
    match = re.search(r"WV0[23]_(\d{14})_([A-Fa-f0-9]+)", producer_id)
    if match:
        return f"{match.group(1)}_{match.group(2).upper()}"
    return str(granule.get("time_start", ""))[:19]


def granule_date(granule):
    value = granule.get("time_start") or granule.get("time_end")
    if not value:
        return None
    return datetime.fromisoformat(value.replace("Z", "+00:00")).date()


def screen_candidate(granule, product, target_date, allowed_months, winter=False):
    acquisition_date = granule_date(granule)
    if acquisition_date is None or acquisition_date.month not in allowed_months:
        return None, "outside requested season"
    umm = load_granule_umm(granule["id"])
    cloud_cover = as_float(granule.get("cloud_cover"))
    if cloud_cover is None:
        cloud_cover = attribute_number(umm, "CLOUDCOVER")
    off_nadir = attribute_number(umm, "OFFNADIR")
    snow_status = snow_free_status(umm) if winter else "not-screened"

    if cloud_cover is None or cloud_cover > MAX_CLOUD_COVER:
        return None, "cloud cover missing or above zero"
    if off_nadir is None:
        return None, "off-nadir angle missing from metadata"
    if off_nadir >= MAX_OFF_NADIR_DEGREES:
        return None, "off-nadir angle at or above 15 degrees"
    if winter and snow_status == "snow-present-or-uncertain":
        return None, "metadata indicates snow or uncertain snow conditions"

    result = dict(granule)
    result["_product"] = product
    result["_acquisition_key"] = acquisition_key(granule)
    result["_acquisition_date"] = acquisition_date.isoformat()
    result["_cloud_cover"] = cloud_cover
    result["_off_nadir_degrees"] = off_nadir
    result["_snow_status"] = snow_status
    result["_date_distance_days"] = abs((acquisition_date - target_date).days)
    return result, None


def best_paired_acquisition(candidates, target_date):
    by_acquisition = {}
    for candidate in candidates:
        by_acquisition.setdefault(candidate["_acquisition_key"], {}).setdefault(
            candidate["_product"], []
        ).append(candidate)
    complete = [
        (key, products) for key, products in by_acquisition.items()
        if "WV2_MSI" in products and "WV2_PAN" in products
    ]
    if not complete:
        return None
    key, products = min(
        complete,
        key=lambda pair: (
            abs((date.fromisoformat(pair[1]["WV2_MSI"][0]["_acquisition_date"])
                 - target_date).days),
            max(item["_off_nadir_degrees"]
                for product_items in pair[1].values() for item in product_items),
        ),
    )
    return {
        "acquisition_key": key,
        "acquisition_date": products["WV2_MSI"][0]["_acquisition_date"],
        "date_distance_days": abs((
            date.fromisoformat(products["WV2_MSI"][0]["_acquisition_date"])
            - target_date
        ).days),
        "granules": [item for product_items in products.values() for item in product_items],
    }


selected_by_site = {}
search_diagnostics = []
for site_name, site in SITES.items():
    print(f"\nSearching {site_name}; target lidar date {site['lidar_date']}")
    primary_products = ["WV2_MSI", "WV2_PAN"]
    raw_by_product = {
        product: search_site_collection(
            site["bbox"], COLLECTIONS[product], site["primary_months"]
        )
        for product in primary_products
    }
    acquisitions = []
    screened = []
    for product, raw_items in raw_by_product.items():
        for granule in raw_items:
            candidate, reason = screen_candidate(
                granule, product, site["lidar_date"], site["primary_months"]
            )
            if candidate:
                screened.append(candidate)
            elif reason != "outside requested season":
                search_diagnostics.append({
                    "site": site_name, "season": "leaf-on", "product": product,
                    "granule_id": granule.get("producer_granule_id", granule.get("id")),
                    "reason": reason,
                })
    primary = best_paired_acquisition(screened, site["lidar_date"])
    if primary:
        acquisitions.append({
            "season": "leaf-on",
            "target_date": site["lidar_date"].isoformat(),
            "acquisition_key": primary["acquisition_key"],
            "acquisition_date": primary["acquisition_date"],
            "date_distance_days": primary["date_distance_days"],
            "granules": primary["granules"],
        })
    else:
        print("  No cloud-free, <15-degree WV2 MSI+PAN leaf-on pair passed screening.")

    if site_name == "SERC":
        winter_screened = []
        for product in primary_products:
            winter_items = search_site_collection(
                site["bbox"], COLLECTIONS[product], WINTER_MONTHS
            )
            for granule in winter_items:
                candidate, reason = screen_candidate(
                    granule, product, site["lidar_date"], WINTER_MONTHS, winter=True
                )
                if candidate:
                    winter_screened.append(candidate)
                elif reason != "outside requested season":
                    search_diagnostics.append({
                        "site": site_name, "season": "leaf-off", "product": product,
                        "granule_id": granule.get("producer_granule_id", granule.get("id")),
                        "reason": reason,
                    })
        winter = best_paired_acquisition(winter_screened, site["lidar_date"])
        if winter:
            acquisitions.append({
                "season": "leaf-off", "target_date": site["lidar_date"].isoformat(),
                "acquisition_key": winter["acquisition_key"],
                "acquisition_date": winter["acquisition_date"],
                "date_distance_days": winter["date_distance_days"],
                "granules": winter["granules"],
            })
            if any(item["_snow_status"] != "snow-free" for item in winter["granules"]):
                print("  NOTE: selected leaf-off pair has unknown snow status; inspect before use.")
        else:
            print("  No winter WV2 MSI+PAN pair passed cloud/off-nadir screening.")

    if site["optional_wv3_swir"]:
        swir_items = search_site_collection(
            site["bbox"], COLLECTIONS["WV3_SWIR"], site["primary_months"]
        )
        swir_candidates = []
        for granule in swir_items:
            candidate, reason = screen_candidate(
                granule, "WV3_SWIR", site["lidar_date"], site["primary_months"]
            )
            if candidate:
                swir_candidates.append(candidate)
            elif reason != "outside requested season":
                search_diagnostics.append({
                    "site": site_name, "season": "leaf-on", "product": "WV3_SWIR",
                    "granule_id": granule.get("producer_granule_id", granule.get("id")),
                    "reason": reason,
                })
        if swir_candidates:
            swir = min(
                swir_candidates,
                key=lambda item: (item["_date_distance_days"], item["_off_nadir_degrees"]),
            )
            acquisitions.append({
                "season": "optional-wv3-swir",
                "target_date": site["lidar_date"].isoformat(),
                "acquisition_key": swir["_acquisition_key"],
                "acquisition_date": swir["_acquisition_date"],
                "date_distance_days": swir["_date_distance_days"],
                "granules": [swir],
            })
        else:
            print("  No Gus Pearson WV3 SWIR candidate passed cloud/off-nadir screening.")

    selected_by_site[site_name] = {"acquisitions": acquisitions}
    for acquisition in acquisitions:
        print(
            f"  {acquisition['season']}: {acquisition['acquisition_date']} "
            f"({acquisition['date_distance_days']} days from lidar target), "
            f"{len(acquisition['granules'])} granule(s)"
        )

if search_diagnostics:
    print(f"\nScreened out {len(search_diagnostics)} granule candidate(s); "
          "details are in search_diagnostics.")

## 2. Authenticate for temporary CSDA S3 access
Provide your CSDA Cognito login and authenticator MFA code. Credentials are temporary and are not written to disk; the downloader will prompt again if they are near expiration.

In [ ]:
class InputParser(html.parser.HTMLParser):
    def __init__(self):
        super().__init__()
        self.values = {}

    def handle_starttag(self, tag, attrs):
        if tag.lower() == "input":
            attributes = dict(attrs)
            name = attributes.get("name")
            if name:
                self.values[name] = attributes.get("value", "")


def fetch_s3_credentials():
    username = input("CSDA Cognito username: ")
    password = getpass.getpass("CSDA Cognito password: ")
    session = requests.Session()

    login_page = session.get(S3_CREDENTIALS_URL, timeout=60)
    login_page.raise_for_status()
    form = InputParser()
    form.feed(login_page.text)
    if "_csrf" not in form.values:
        raise RuntimeError("Could not find CSRF token on the CSDA login page")

    login_data = {
        "_csrf": form.values["_csrf"],
        "username": username,
        "password": password,
    }
    if form.values.get("cognitoAsfData"):
        login_data["cognitoAsfData"] = form.values["cognitoAsfData"]
    login_page = session.post(login_page.url, data=login_data, timeout=60)
    login_page.raise_for_status()

    mfa_form = InputParser()
    mfa_form.feed(login_page.text)
    if "_csrf" not in mfa_form.values:
        raise RuntimeError("CSDA did not present an MFA form; inspect the login response")
    mfa_data = {
        "_csrf": mfa_form.values["_csrf"],
        "authentication_code": input("CSDA authenticator MFA code: ").strip(),
        "cognitoAsfData": mfa_form.values.get("cognitoAsfData", ""),
    }
    response = session.post(
        login_page.url, data=mfa_data, allow_redirects=False, timeout=60
    )
    if response.is_redirect:
        response = session.get(urljoin(response.url, response.headers["Location"]), timeout=60)
    response.raise_for_status()
    if "json" not in response.headers.get("content-type", "").lower():
        response = session.get(S3_CREDENTIALS_URL, timeout=60)
        response.raise_for_status()
    credentials = response.json()
    required = {"accessKeyId", "secretAccessKey", "sessionToken"}
    if not required.issubset(credentials):
        raise RuntimeError("CSDA response did not contain the expected temporary S3 credentials")
    return credentials


def build_s3_client(credentials):
    return boto3.client(
        "s3",
        aws_access_key_id=credentials["accessKeyId"],
        aws_secret_access_key=credentials["secretAccessKey"],
        aws_session_token=credentials["sessionToken"],
        region_name="us-west-2",
        config=Config(retries={"max_attempts": 5, "mode": "standard"}),
    )


def refresh_s3_client():
    global credentials, s3_client, credential_expiration
    credentials = fetch_s3_credentials()
    expiration = credentials.get("expiration")
    credential_expiration = (
        datetime.fromisoformat(expiration.replace("Z", "+00:00"))
        if expiration else datetime.now(timezone.utc) + timedelta(minutes=55)
    )
    if credential_expiration.tzinfo is None:
        credential_expiration = credential_expiration.replace(tzinfo=timezone.utc)
    s3_client = build_s3_client(credentials)


def ensure_s3_client():
    if credential_expiration <= datetime.now(timezone.utc) + timedelta(minutes=5):
        print("Temporary credentials are expiring; authenticate again to continue downloads.")
        refresh_s3_client()
    return s3_client


refresh_s3_client()
print("Temporary CSDA S3 credentials acquired; credentials are short-lived.")

## 3. Download granules and save manifest
Downloads NTF/TIFF imagery and available XML/RPB/IMD sidecars into site/year/granule folders.

In [ ]:
def s3_data_links(granule: dict) -> list[tuple[str, str]]:
    'Return S3 bucket/key pairs for NTF data and available sidecars.'
    files = {}
    for link in granule.get("links", []):
        href = link.get("href", "")
        if not href.startswith("s3://"):
            continue
        parsed = urlparse(href)
        suffix = Path(parsed.path).suffix.lower()
        if suffix not in DOWNLOAD_SUFFIXES:
            continue
        key = parsed.path.lstrip("/")
        if parsed.netloc and key:
            files[(parsed.netloc, key)] = (parsed.netloc, key)
    return list(files.values())


def safe_name(value: str) -> str:
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", value).strip("._")
    return name or "granule"


def download_site_granules() -> list[dict]:
    manifest_rows = []
    for site_name, site_result in selected_by_site.items():
        for acquisition in site_result["acquisitions"]:
            selected_year = int(acquisition["acquisition_date"][:4])
            for granule in acquisition["granules"]:
                granule_id = (
                    granule.get("producer_granule_id")
                    or granule.get("title")
                    or granule.get("id")
                )
                granule_dir = (
                    OUTPUT_DIR / safe_name(site_name) / acquisition["season"]
                    / str(selected_year) / safe_name(granule_id)
                )
                granule_dir.mkdir(parents=True, exist_ok=True)
                links = s3_data_links(granule)
                if not any(
                    Path(key).suffix.lower() in {".ntf", ".tif", ".tiff"}
                    for _, key in links
                ):
                    raise RuntimeError(
                        f"No NTF/TIFF S3 data link found for CMR granule {granule_id}."
                    )

                local_files = []
                for bucket, key in links:
                    filename = safe_name(posixpath.basename(key))
                    destination = granule_dir / filename
                    if destination.exists() and destination.stat().st_size > 0:
                        print(f"Exists: {destination}")
                    else:
                        print(f"Downloading s3://{bucket}/{key}")
                        for attempt in range(2):
                            try:
                                ensure_s3_client().download_file(bucket, key, str(destination))
                                break
                            except Exception as error:
                                error_response = getattr(error, "response", {})
                                aws_error = error_response.get("Error", {})
                                error_code = aws_error.get("Code", type(error).__name__)
                                error_message = aws_error.get("Message", str(error))
                                expired_token_codes = {
                                    "ExpiredToken", "ExpiredTokenException",
                                    "InvalidToken", "TokenRefreshRequired",
                                }
                                if attempt == 0 and error_code in expired_token_codes:
                                    print("S3 token expired; refreshing credentials and retrying once.")
                                    refresh_s3_client()
                                    continue

                                if error_code in {"AccessDenied", "403", "AllAccessDisabled"}:
                                    advice = (
                                        " The AWS principal is not authorized by the CSDA bucket policy. "
                                        "Setting region_name='us-west-2' selects the endpoint but does not "
                                        "grant access; use an approved CSDA in-region AWS role/account."
                                    )
                                elif error_code in expired_token_codes:
                                    advice = " Refresh CSDA S3 credentials and retry the download."
                                else:
                                    advice = " Check network access, bucket permissions, and credential validity."
                                raise RuntimeError(
                                    f"S3 download failed for s3://{bucket}/{key}. "
                                    f"AWS error {error_code}: {error_message}.{advice}"
                                ) from error
                    local_files.append(str(destination))

                manifest_rows.append({
                    "site": site_name,
                    "season": acquisition["season"],
                    "selected_year": selected_year,
                    "month": int(acquisition["acquisition_date"][5:7]),
                    "target_lidar_date": acquisition["target_date"],
                    "acquisition_date": acquisition["acquisition_date"],
                    "date_distance_days": acquisition["date_distance_days"],
                    "product": granule.get("_product"),
                    "granule_id": granule_id,
                    "cloud_cover": granule.get("_cloud_cover"),
                    "off_nadir_degrees": granule.get("_off_nadir_degrees"),
                    "snow_status": granule.get("_snow_status"),
                    "s3_uris": ";".join(f"s3://{bucket}/{key}" for bucket, key in links),
                    "local_files": ";".join(local_files),
                })
    return manifest_rows


manifest_rows = download_site_granules()
manifest_fields = [
    "site", "season", "selected_year", "month", "target_lidar_date",
    "acquisition_date", "date_distance_days", "product", "granule_id",
    "cloud_cover", "off_nadir_degrees", "snow_status", "s3_uris", "local_files",
]
with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as manifest_file:
    writer = csv.DictWriter(manifest_file, fieldnames=manifest_fields)
    writer.writeheader()
    writer.writerows(manifest_rows)

print(f"\nDownloaded {len(manifest_rows)} selected granule(s). Manifest: {MANIFEST_PATH}")
for row in manifest_rows:
    print(
        f"{row['site']} {row['season']} {row['acquisition_date']} "
        f"[{row['product']}]: {row['granule_id']}"
    )